# Forecast future speed, heading and altitude

This is the current forecasting task for pilots using `change_trajectory` and `launch`.
GRU and Transformer models regress future physical states at +1, +2 and +5 seconds.
Heading means horizontal course clockwise from north, derived from NWU velocity.
Sine/cosine heading residuals avoid a discontinuity at north. The model receives
only current/past observer frames; target actions and skill labels are excluded.

Train fresh pilots with notebook 05 or 06, then collect B01 observer episodes using
`generate_tactical_corpus.py --perspective tactical --job <completed-population>`.
Set `CORPUS` below to the packaged corpus. No dataset download or full training starts
until its corresponding cell is run. Earlier classification notebooks are historical.


In [ ]:
import json
import os
from pathlib import Path
import sys
import torch

ROOT = Path.cwd().resolve()
if not (ROOT / "bvr_behavior_prediction").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from bvr_behavior_prediction.forecasting.data import prepare_dataset, read_json
from bvr_behavior_prediction.forecasting.train import train_model
from bvr_behavior_prediction.forecasting.flight_state import FlightStatePredictor

CORPUS = Path(os.getenv("BVR_TRAJECTORY_CORPUS", ROOT / "shared/corpora/trajectory-pilots-v4"))
DATA = ROOT / "artifacts/trajectory_forecasting/data_v1"
OUTPUT = ROOT / "artifacts/trajectory_forecasting/runs_v1"
CONFIG = read_json(ROOT / "configs/training/trajectory_forecasting.json")
CONFIG["training"]["device"] = os.getenv("BVR_TRAIN_DEVICE", "cuda" if torch.cuda.is_available() else "cpu")
print({"corpus": str(CORPUS), "data": str(DATA), "device": CONFIG["training"]["device"]})


## Prepare grouped histories

Complete scenario groups stay in one split across pilots. Reserved test episodes
are not loaded into preparation. Feature/target scaling uses training groups only.


In [ ]:
if not (DATA / "data_manifest.json").exists():
    manifest = prepare_dataset(CORPUS, DATA, target="flight_state", **CONFIG["data"])
else:
    manifest = read_json(DATA / "data_manifest.json")
print(json.dumps(manifest["counts"], indent=2))


## Train and compare

This cell trains both architectures. Shorten `epochs` in `CONFIG["training"]` for
an installation check. Validation selects the best checkpoint using equal-episode
speed MAE (m/s), circular heading MAE (degrees), and altitude MAE (m), relative to
the constant-state baseline. The held-out test set remains reserved.


In [ ]:
results = {}
for architecture in ("gru", "transformer"):
    results[architecture] = train_model(ROOT, DATA, OUTPUT / architecture,
                                        architecture, CONFIG["training"])
print(json.dumps(results, indent=2))


In [ ]:
import pandas as pd
metrics = pd.concat([pd.read_csv(OUTPUT / name / "validation_metrics.csv")
                     for name in ("gru", "transformer")], ignore_index=True)
display(metrics.query("pilot_id == 'population'"))
# Inference accepts at least 56 consecutive raw 10 Hz B01 frames:
# predictor = FlightStatePredictor(OUTPUT / "transformer/best.pt")
# future = predictor.predict(raw_observer_frames)
# future: horizons_s, speed_mps, heading_deg and altitude_m (each length 3).
